In [2]:
from pathlib import Path
import pandas as pd
from openpyxl.styles import Alignment, Font, PatternFill


# 1. Set the folder containing your four ERS CSV files

BASE = Path(
    "/Users/qandeelmaqsood/Desktop/UIT/Thesis 2026-2027/"
    "ERS/elektronisk-rapportering-ers-2025"
)

if not BASE.is_dir():
    raise FileNotFoundError(f"Folder not found: {BASE}")



# 2. Find the DEP, POR, DCA and TRA files automatically

FILES = {}

for kind in ["DEP", "POR", "DCA", "TRA"]:
    matches = [
        file
        for file in BASE.iterdir()
        if file.suffix.lower() == ".csv"
        and f"-{kind.lower()}" in file.name.lower()
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected one {kind} CSV, but found {len(matches)}: "
            f"{[file.name for file in matches]}"
        )

    FILES[kind] = matches[0]


# 
# 3. Helper functions

def parse_time(column):
    """Convert date/time text into Python timestamps."""
    return pd.to_datetime(
        column,
        format="%d.%m.%Y %H:%M:%S",
        errors="coerce",
    )


def unique_values(column):
    """Put unique values into one cell, separated by semicolons."""
    values = {
        str(value).strip()
        for value in column.dropna()
        if str(value).strip()
    }
    return "; ".join(sorted(values))


def code_and_name(df, code_column, name_column):
    """Combine a code and its description."""
    return df[[code_column, name_column]].apply(
        lambda row: " | ".join(
            str(value).strip()
            for value in row
            if pd.notna(value) and str(value).strip()
        ),
        axis=1,
    )


# 4. Read and prepare the four files

data = {}

for kind, file in FILES.items():
    print(f"Reading {kind}: {file.name}")

    # Read IDs as text to preserve their formatting.
    df = pd.read_csv(
        file,
        sep=";",
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
    )

    # Remove exact duplicate rows.
    df = df.drop_duplicates()

    # Vessel ID connects records across the four files.
    df["vessel_id"] = df["Fartøy ID"].str.strip()
    df = df[df["vessel_id"] != ""].copy()
    
    #Time of notification
    df["report_time"] = parse_time(df["Meldingstidspunkt"])

    if kind == "DCA":
        # Keep all detail rows from the latest DCA message version.
        df["version"] = pd.to_numeric(
            df["Meldingsversjon"],
            errors="coerce",
        ).fillna(0)

        message_keys = [
            "vessel_id",
            "Meldingsår",
            "Meldingsnummer",
        ]

        latest_version = df.groupby(
            message_keys
        )["version"].transform("max")

        df = df[df["version"] == latest_version].copy()

        # Fishing-operation timestamps.
        df["operation_start"] = parse_time(df["Starttidspunkt"])
        df["operation_end"] = parse_time(df["Stopptidspunkt"])

        # Select records using operation end.
        # If missing, use operation start, then report time.
        df["event_time"] = (
            df["operation_end"]
            .fillna(df["operation_start"])
            .fillna(df["report_time"])
        )

        # Convert catch weights from text into numbers.
        df["catch_kg"] = pd.to_numeric(
            df["Rundvekt"].str.replace(",", ".", regex=False),
            errors="coerce",
        )

        # Keep catch weight for COD; other species contribute zero.
        df["cod_kg"] = df["catch_kg"].where(
            df["Art FAO (kode)"].str.strip().str.upper().eq("COD"),
            0,
        )

        df["area"] = code_and_name(
            df,
            "Hovedområde start (kode)",
            "Hovedområde start",
        )

        df["gear"] = code_and_name(
            df,
            "Redskap FDIR (kode)",
            "Redskap FDIR",
        )

        df["species"] = code_and_name(
            df,
            "Art - FDIR (kode)",
            "Art - FDIR",
        )

    elif kind == "DEP":
        df["event_time"] = parse_time(df["Avgangstidspunkt"])

    elif kind == "POR":
        df["event_time"] = parse_time(df["Ankomsttidspunkt"])

    elif kind == "TRA":
        # Use the reported transshipment time.
        df["event_time"] = parse_time(df["Omlastingstidspunkt"])

    # Keep only the columns needed for the summary.
    columns = [
        "vessel_id",
        "Fartøynavn",
        "Fartøynavn (ERS)",
        "Melding ID",
        "report_time",
        "event_time",
    ]

    if kind in ["DEP", "POR"]:
        # Retain port codes only.
        columns += ["Havn (kode)"]

    if kind == "DCA":
        columns += [
            "operation_start",
            "operation_end",
            "catch_kg",
            "cod_kg",
            "area",
            "gear",
            "species",
        ]

    data[kind] = df[columns].copy()



# 5. Group each file by vessel ID

groups = {
    kind: dict(tuple(df.groupby("vessel_id")))
    for kind, df in data.items()
}

# Include every vessel appearing in any of the four files.
vessel_ids = sorted(
    set().union(*(set(df["vessel_id"]) for df in data.values()))
)

print(f"\nProcessing {len(vessel_ids)} vessels...")

rows = []


# 6. Create one summary row for each vessel

for vessel_id in vessel_ids:

    # Get this vessel's records from each file.
    vessel = {
        kind: groups[kind].get(vessel_id, data[kind].iloc[:0])
        for kind in data
    }

    # Count each distinct departure/arrival timestamp once.
    dep = (
        vessel["DEP"]
        .dropna(subset=["event_time"])
        .sort_values(["event_time", "report_time"])
        .drop_duplicates("event_time", keep="last")
    )

    por = (
        vessel["POR"]
        .dropna(subset=["event_time"])
        .sort_values(["event_time", "report_time"])
        .drop_duplicates("event_time", keep="last")
    )

    dca = vessel["DCA"]
    tra = vessel["TRA"]

    # Start of the period: first recorded departure.
    first_departure = dep.iloc[0] if not dep.empty else None

    departure_time = (
        first_departure["event_time"]
        if first_departure is not None
        else pd.NaT
    )

    # End of the period: last reported arrival after departure.
    valid_arrivals = (
        por[por["event_time"] >= departure_time]
        if pd.notna(departure_time)
        else por.iloc[:0]
    )

    last_arrival = (
        valid_arrivals.iloc[-1]
        if not valid_arrivals.empty
        else None
    )

    arrival_time = (
        last_arrival["event_time"]
        if last_arrival is not None
        else pd.NaT
    )

    # Both timestamps are needed to define the period.
    has_period = (
        pd.notna(departure_time)
        and pd.notna(arrival_time)
    )

    if has_period:
        # Select fishing records within the period.
        operations = dca[
            dca["event_time"].between(departure_time, arrival_time)
        ]

        # Select transshipment records within the same period.
        transfers = tra[
            tra["event_time"].between(departure_time, arrival_time)
        ]

        departures_in_period = dep[
            dep["event_time"].between(departure_time, arrival_time)
        ]

        # Count departures followed by an arrival before
        # the next departure.
        complete_journeys = 0
        departure_times = dep["event_time"].tolist()

        for index, start in enumerate(departure_times):
            if start > arrival_time:
                break

            next_departure = (
                departure_times[index + 1]
                if index + 1 < len(departure_times)
                else pd.Timestamp.max
            )

            matching_arrivals = valid_arrivals[
                (valid_arrivals["event_time"] >= start)
                & (valid_arrivals["event_time"] < next_departure)
            ]

            if not matching_arrivals.empty:
                complete_journeys += 1

    else:
        # Leave period-based results blank if dates are missing.
        operations = dca.iloc[:0]
        transfers = tra.iloc[:0]
        departures_in_period = dep.iloc[:0]
        complete_journeys = None

    # Sort unique transshipment timestamps.
    # Dates and times correspond by position in their cells.
    transfer_times = (
        transfers["event_time"]
        .dropna()
        .drop_duplicates()
        .sort_values()
    )

    # Get the latest available vessel name.
    name_records = pd.concat(
        [
            df[["report_time", "Fartøynavn", "Fartøynavn (ERS)"]]
            for df in vessel.values()
        ],
        ignore_index=True,
    ).sort_values("report_time", na_position="first")

    names = name_records["Fartøynavn"].where(
        name_records["Fartøynavn"].str.strip().ne(""),
        name_records["Fartøynavn (ERS)"],
    )

    names = names[names.str.strip().ne("")]
    vessel_name = names.iloc[-1] if not names.empty else ""

    # List the years covered by the period.
    if has_period:
        years = range(departure_time.year, arrival_time.year + 1)
    else:
        years = sorted(
            name_records["report_time"].dropna().dt.year.unique()
        )

    # Build the vessel's summary row.
    rows.append({
        "Vessel ID": vessel_id,
        "Vessel name": vessel_name,
        "Year": "; ".join(str(year) for year in years),

        "Journey": (
            f"{departure_time:%Y-%m-%d %H:%M} → "
            f"{arrival_time:%Y-%m-%d %H:%M}"
            if has_period else ""
        ),

        "Completed journeys": complete_journeys,

        "Departures within period": (
            len(departures_in_period) if has_period else None
        ),

        "Total catch (kg)": operations["catch_kg"].sum(min_count=1),
        "Area": unique_values(operations["area"]),

        "First operation start (UTC)": (
            operations["operation_start"].min()
        ),

        "Last operation end (UTC)": (
            operations["operation_end"].max()
        ),

        "Gear used": unique_values(operations["gear"]),
        "Total COD (kg)": operations["cod_kg"].sum(min_count=1),
        "Species FDIR": unique_values(operations["species"]),

        "Arrival port code": (
            last_arrival["Havn (kode)"]
            if last_arrival is not None else ""
        ),

        "Last arrival time (UTC)": arrival_time,

        "Departure port code": (
            first_departure["Havn (kode)"]
            if first_departure is not None else ""
        ),

        "First departure time (UTC)": departure_time,

        "Transshipment date (UTC)": "; ".join(
            transfer_times.dt.strftime("%Y-%m-%d")
        ),

        "Transshipment time (UTC)": "; ".join(
            transfer_times.dt.strftime("%H:%M:%S")
        ),

        "TRA messages in file": tra["Melding ID"].nunique(),

        "TRA messages within period": (
            transfers["Melding ID"].nunique()
            if has_period else None
        ),
    })


# 7. Create the final table

summary = pd.DataFrame(rows)

# Ensure unwanted columns are absent.
summary = summary.drop(
    columns=[
        "Window status",
        "Arrival port",
        "Departure port",
        "Transshipment status",
    ],
    errors="ignore",
)

# Verify that each vessel appears only once.
assert summary["Vessel ID"].is_unique


# 8. Save the table as an Excel workbook
# Close the existing workbook in Excel before running.

output = BASE / "vessel_summary.xlsx"

with pd.ExcelWriter(
    output,
    engine="openpyxl",
    datetime_format="yyyy-mm-dd hh:mm:ss",
) as writer:

    summary.to_excel(
        writer,
        sheet_name="Vessel Summary",
        index=False,
    )

    sheet = writer.sheets["Vessel Summary"]

    # Keep headings and vessel identifiers visible while scrolling.
    sheet.freeze_panes = "C2"
    sheet.auto_filter.ref = sheet.dimensions
    sheet.row_dimensions[1].height = 32

    # Format the header.
    for cell in sheet[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill(
            fill_type="solid",
            fgColor="203864",
        )
        cell.alignment = Alignment(
            wrap_text=True,
            vertical="top",
        )

    # Set readable column widths.
    for column in sheet.columns:
        heading = column[0].value
        width = 25

        if heading in ["Area", "Gear used", "Species FDIR"]:
            width = 45
        elif heading == "Journey":
            width = 42
        elif heading in [
            "Transshipment date (UTC)",
            "Transshipment time (UTC)",
        ]:
            width = 32

        sheet.column_dimensions[column[0].column_letter].width = width

    # Allow long lists to wrap within cells.
    for row in sheet.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(
                wrap_text=True,
                vertical="top",
            )


# 9. Showing the results

print(f"\nSaved {len(summary)} vessel rows to:\n{output}")
display(summary.head(20))

Reading DEP: elektronisk-rapportering-ers-2025-avgangsmelding-dep.csv
Reading POR: elektronisk-rapportering-ers-2025-ankomstmelding-por.csv
Reading DCA: elektronisk-rapportering-ers-2025-fangstmelding-dca.csv
Reading TRA: elektronisk-rapportering-ers-2025-overforingsmelding-tra.csv

Processing 427 vessels...

Saved 427 vessel rows to:
/Users/qandeelmaqsood/Desktop/UIT/Thesis 2026-2027/ERS/elektronisk-rapportering-ers-2025/vessel_summary.xlsx


,Vessel ID,Vessel name,Year,Journey,Completed journeys,Departures within period,Total catch (kg),Area,First operation start (UTC),Last operation end (UTC),...,Total COD (kg),Species FDIR,Arrival port code,Last arrival time (UTC),Departure port code,First departure time (UTC),Transshipment date (UTC),Transshipment time (UTC),TRA messages in file,TRA messages within period
0,1959006996,REINSBÅEN,2025,2025-01-04 12:38 → 2025-12-15 09:45,98.0,98.0,48328.0,04 | Vest-Finnmark; 05 | Røstbanken til Malang...,2025-01-04 17:07:00,2025-12-13 16:25:00,...,41796.0,1021 | Brosme; 1022 | Torsk; 1023 | Lange; 102...,NOKBV,2025-12-15 09:45:00,NOSSO,2025-01-04 12:38:00,,,0,0.0
1,1967005695,BUHOLMEN,2025,2025-03-28 03:20 → 2025-06-10 07:35,27.0,27.0,71376.0,05 | Røstbanken til Malangsgrunnen,2025-03-27 01:51:00,2025-06-09 17:07:00,...,54100.0,1021 | Brosme; 1022 | Torsk; 1023 | Lange; 102...,NOVEY,2025-06-10 07:35:00,NOVEY,2025-03-28 03:20:00,,,0,0.0
2,1967011839,TOROLV,2025,2025-01-06 05:28 → 2025-05-30 10:00,26.0,28.0,69556.0,04 | Vest-Finnmark; 05 | Røstbanken til Malang...,2025-01-06 07:02:00,2025-05-29 23:20:00,...,53277.0,"0529 | Skate, uspesifisert; 1022 | Torsk; 1023...",NOSOM,2025-05-30 10:00:00,NOVEO,2025-01-06 05:28:00,,,0,0.0
3,1969000787,NORDSTAR,2025,2025-01-09 22:20 → 2025-12-09 02:30,14.0,14.0,6221217.0,00 | Vestfjorden (Lofoten); 04 | Vest-Finnmark...,2025-01-10 14:52:00,2025-12-07 17:34:00,...,197562.0,0742 | Vassild; 1022 | Torsk; 1023 | Lange; 10...,NOAES,2025-12-09 02:30:00,NOAES,2025-01-09 22:20:00,,,0,0.0
4,1970005993,GILL,2025,2025-01-30 05:51 → 2025-05-02 16:00,22.0,22.0,18630.0,00 | Vestfjorden (Lofoten); 04 | Vest-Finnmark,2025-01-30 03:52:00,2025-05-02 15:52:00,...,18150.0,1022 | Torsk; 1032 | Sei,NOFKS,2025-05-02 16:00:00,NOHAV,2025-01-30 05:51:00,,,0,0.0
5,1970007571,ANN BRITA,2025,2025-01-01 07:00 → 2025-08-05 21:55,69.0,69.0,203490.0,00 | Vestfjorden (Lofoten); 03 | Øst-Finnmark;...,2025-01-02 18:00:00,2025-08-05 14:15:00,...,84478.0,1021 | Brosme; 1022 | Torsk; 1023 | Lange; 102...,NOVVY,2025-08-05 21:55:00,NOMSK,2025-01-01 07:00:00,,,0,0.0
6,1971010932,NY-MÅTIND,2025,2025-01-05 09:04 → 2025-09-14 13:19,28.0,28.0,19714.0,00 | Vestfjorden (Lofoten),2025-01-05 09:55:00,2025-09-14 11:52:00,...,15670.0,1022 | Torsk; 1023 | Lange; 1027 | Hyse; 1032 ...,NOMSK,2025-09-14 13:19:00,NOMSK,2025-01-05 09:04:00,,,0,0.0
7,1973004689,KATO,2025,2025-04-14 18:06 → 2025-09-08 10:00,13.0,13.0,148330.0,03 | Øst-Finnmark; 05 | Røstbanken til Malangs...,2025-04-21 10:26:00,2025-09-02 13:00:00,...,0.0,2729 | Vågehval,NOAES,2025-09-08 10:00:00,NOSVR,2025-04-14 18:06:00,,,0,0.0
8,1973011832,ALF SENIOR,2025,2025-01-23 06:14 → 2025-12-16 12:29,17.0,17.0,38195.0,04 | Vest-Finnmark; 07 | Storegga-Frøyabanken,2025-01-23 09:04:00,2025-12-16 09:26:00,...,37290.0,"0412 | Pigghå; 0529 | Skate, uspesifisert; 102...",NOHRS,2025-12-16 12:29:00,NOHRS,2025-01-23 06:14:00,,,0,0.0
9,1974008967,FLØSUND,2025,2025-02-18 13:53 → 2025-02-18 19:15,1.0,1.0,25.0,08 | Eigersundbanken,2025-02-18 14:47:00,2025-02-18 16:44:00,...,0.0,2524 | Dypvannsreke,NOSKU,2025-02-18 19:15:00,NOSKU,2025-02-18 13:53:00,,,0,0.0
